# Chat with vision models

**If you're looking for the web application, check the src/ folder.**

This notebook is just provided for manual experimentation with the vision model.

## Authenticate to OpenAI

The following code connects to OpenAI, either using an Azure OpenAI account or local Ollama model. See the README for instructions on configuring the `.env` file.

In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

endpoint = os.environ["FOUNDRY_MODELS_ENDPOINT"] + "/openai/v1"
deployment_name = os.environ["FOUNDRY_OPENAI_DEPLOYMENT"]
api_key = os.environ["FOUNDRY_API_KEY"]

openai_client = OpenAI(
    base_url=endpoint,
    api_key=api_key,
)

model_name = deployment_name

print(f"Using model {model_name}")

Using model Mistral-Large-3


## Send an image by URL

In [22]:
input_items = [
    {
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Is this a unicorn?"},
            {
                "image_url": "https://raw.githubusercontent.com/Azure-Samples/openai-chat-vision-quickstart/main/notebooks/ur.jpg",
                "type": "input_image",
            },
        ],
    }
]
response = openai_client.responses.create(model=model_name, input=input_items, store=False)

print(response.output_text)

BadRequestError: Error code: 400 - {'error': {'message': "Model 'Mistral-Large-3' does not support image inputs. Try again with a vision model. To find out which models are vision-enabled, visit our our models documentation: https://platform.openai.com/docs/models", 'type': 'invalid_request_error', 'param': 'input', 'code': None}}

## Send an image by Data URI



In [ ]:
import base64


def open_image_as_base64(filename):
    with open(filename, "rb") as image_file:
        image_data = image_file.read()
    image_base64 = base64.b64encode(image_data).decode("utf-8")
    return f"data:image/png;base64,{image_base64}"

In [ ]:
response = openai_client.responses.create(
    model=model_name,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "are these alligators or crocodiles?"},
                {"type": "input_image", "image_url": open_image_as_base64("mystery_reptile.png")},
            ],
        }
    ],
)

print(response.output_text)

They look more like **crocodiles** than alligators.

Main clues from the image:
- Their snouts appear relatively **narrow and V-shaped**, which is typical of crocodiles.
- Alligators usually have a **broader, more rounded U-shaped snout**.
- The lighter gray/tan coloration also fits many crocodile species, though color alone isn’t definitive.

So: **likely crocodiles**, probably juveniles or young individuals.


## Use cases for image analysis

### Accessibility

#### Assistance for vision-impaired

In [ ]:
response = openai_client.responses.create(
    model=model_name,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "is there anything good for vegans on this menu?"},
                {"type": "input_image", "image_url": open_image_as_base64("menu.png")},
            ],
        }
    ],
)

print(response.output_text)

Yes — there are a few good vegan-friendly options, mostly with small confirmations/modifications:

**Best bets as-is or likely vegan**
- **Insalata Di Mista** — seasonal greens with house vinaigrette. Confirm vinaigrette has no honey/dairy.
- **Panzanella con Fagioli** — tomato/bread salad with cannellini beans, cucumber, avocado, vinaigrette. Confirm bread/vinaigrette are vegan.
- **Spinaci Soffritti** — spinach with lemon and garlic. Ask if it’s cooked in olive oil, not butter.
- **Zuppa di Fagioli** — Tuscan beans and pasta. Ask if it’s made with vegetable stock and no parmesan/meat.

**Good with modifications**
- **Bruschetta Trio** — ask for only the cherry tomato/caper/garlic/basil bruschetta, and skip the goat cheese and salmon versions.
- **Spaghetti Ortolano** — ask for **no goat cheese** and confirm the pasta is egg-free.
- Possibly **Dal Forno / pizza** — ask if they can do a cheeseless vegetable pizza with vegan dough.

I’d avoid the risotto unless they can make it without 

#### Automated image captioning

In [ ]:
response = openai_client.responses.create(
    model=model_name,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Suggest an alt text for this image"},
                {"type": "input_image", "image_url": open_image_as_base64("azure_arch.png")},
            ],
        }
    ],
)

print(response.output_text)

Azure resource diagram showing a Container App connected to a Container Apps Environment and a Managed Identity, alongside related resources: Azure AI services, Log Analytics workspace, Container Registry, and Key Vault.


### Business process automation

#### Insurance claim processing

In [ ]:
response = openai_client.responses.create(
    model=model_name,
    input=[
        {
            "role": "system",
            "content": (
                "You are an AI assistant that helps auto insurance companies process claims."
                "You accept images of damaged cars that are submitted with claims, and you are able to make judgments "
                "about the causes of automobile damage, and the validity of claims regarding that damage."
            ),
        },
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Claim states that this damage is due to hail. Is it valid?"},
                {"type": "input_image", "image_url": open_image_as_base64("dented_car.jpg")},
            ],
        },
    ],
)

print(response.output_text)

No — the claim that this damage is due to hail does **not** appear valid.

The vehicle shows **major front-end impact damage**, including:

- Severely crumpled hood with large deformation
- Displaced/misaligned front grille and fascia
- Damage concentrated at the front center/right area
- Panel buckling consistent with a collision or impact with an object

Hail damage typically appears as **many small, round dents** across horizontal surfaces like the hood, roof, and trunk, without major structural displacement. This damage is much more consistent with a **front-end collision**, not hail.


#### Graph analysis

In [ ]:
input_items = [
    {
        "role": "user",
        "content": [
            {"type": "input_text", "text": "What zone are we losing the most trees in?"},
            {
                "image_url": open_image_as_base64("tree_cover_loss.png"),
                "type": "input_image",
            },
        ],
    }
]
response = openai_client.responses.create(model=model_name, input=input_items, store=False)

print(response.output_text)

We’re losing the most trees in the **Tropical zone**. It consistently makes up the largest share of annual global tree cover loss in the chart.


#### Table analysis

In [ ]:
response = openai_client.responses.create(
    model=model_name,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "What's the cheapest plant?"},
                {"type": "input_image", "image_url": open_image_as_base64("page_0.png")},
            ],
        }
    ],
)

print(response.output_text)

The cheapest plant listed is **Agrostis pallens** — **Thingrass**, priced at **$0.58**.


#### Appliance support

In [ ]:
response = openai_client.responses.create(
    model=model_name,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "How do I set this to wash the dishes quickly?"},
                {"type": "input_image", "image_url": open_image_as_base64("dishwasher.png")},
            ],
        }
    ],
)

print(response.output_text)

To wash quickly:

1. Load dishes and add detergent.
2. Press **On/Off**.
3. Press the **Quick 45°** program button — it’s the button labeled **“Quick 45°”** between **Glass 40°** and **Pre-Rinse**.
4. If you want it even faster, press **VarioSpeed** if available/compatible.
5. Press **Start** on the far right.
6. Close the dishwasher door.

Note: The **Check Water** light appears to be on in the photo. If it stays on, make sure the water tap/supply is open and the inlet hose isn’t kinked.
